# Latent faktormodellering af kreditrisiko med PROC HPPLS

## Resumé

En detailbank ønsker at forudsige tre korrelerede kreditrisiko-udfald — kortudnyttelse, udvikling i gældskvote og en misligholdelsessandsynligheds-proxy — ud fra et bredt sæt af stærkt kollineære kreditoplysnings- og makroøkonomiske prædiktorer. Almindelig regression bryder sammen under denne kollinearitet, så denne notebook bruger **PROC HPPLS** (high-performance partial least squares) til at udtrække en håndfuld latente faktorer, der tilsammen forklarer prædiktorerne og alle tre respons-variable, og validerer derefter antallet af faktorer med en van der Voet-krydsvalideringstest på et udeladt porteføljesegment.

## Datakilder

Alle data genereres syntetisk direkte i notebooken med `call streaminit(20240531)` — ingen eksterne filer eller netværksadgang.

| Datasæt | Rækker | Variabel | Rolle | Beskrivelse |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Unik kunde-nøgle, som følger med til det scorede output |
| | | `Segment` | CLASS-prædiktor | Porteføljesegment: Retail (privat), Affluent (velhavende), Business (erhverv) |
| | | `b1`–`b12` | Prædiktorer | 12 kollineære månedlige kreditoplysnings-adfærdssignaler |
| | | `RatePctChg` | Prædiktor | Kundens eksponering over for renteændringer |
| | | `InquiryCount` | Prædiktor | Antal seneste hårde kreditforespørgsler |
| | | `Utilization` | Respons | Udnyttelsesgrad af revolverende kredit (%) |
| | | `DTIChange` | Respons | Ændring i gældskvote |
| | | `DefaultProp` | Respons | Misligholdelsessandsynligheds-proxy (0–1) |
| | | `Role` | Partition | TRAIN (~75%) / TEST (~25%) valideringsflag |

# Latent faktormodellering af kreditrisiko med PROC HPPLS

Banker står ofte over for problemet med **brede, kollineære prædiktorer**: snesevis af månedlige kreditoplysningsattributter, makroøkonomiske eksponeringer og adfærdssignaler, der bevæger sig sammen, og som bruges til at forudsige flere risiko-udfald, der selv er korrelerede. Mindste kvadraters metode er ustabil her, fordi prædiktormatricen er næsten singulær.

**Partial least squares (PLS)** løser dette ved at udtrække et lille antal latente faktorer fra *krydskovariansen* mellem prædiktorerne (X) og respons-variablene (Y), så faktorerne er tilpasset til at forudsige udfaldene — ikke blot til at opsummere X. `PROC HPPLS` er den højtydende modstykke til `PROC PLS`, der tilføjer multitrådet udførelse, datapartitionering til validering og van der Voet-randomiseringstesten til valg af antallet af faktorer.

I denne notebook opbygger vi en enkelt PLS-model, der samtidig forudsiger tre korrelerede kreditrisiko-respons-variable, og bruger et udeladt valideringssegment til at bekræfte, hvor mange latente faktorer dataene reelt understøtter.

## Trin 1 — Generer et syntetisk kreditrisiko-panel

Vi simulerer 600 kunder. To uobserverede latente drivere (`stress` og `tenure`) genererer tolv kollineære kreditoplysningssignaler `b1`–`b12` samt rente- og forespørgselseksponeringer — netop den stærkt korrelerede struktur, PLS er designet til. De tre respons-variable (`Utilization`, `DTIChange`, `DefaultProp`) er forskellige lineære kombinationer af de samme drivere, så de er også indbyrdes korrelerede. Et `Role`-flag udelader cirka en fjerdedel af porteføljen som et valideringssegment.

In [1]:
data credit;
   CALL streaminit(20240531);
   LÆNGDE Segment $8 Role $5;
   GØR CustomerID = 1 TIL 600;
      /* kundesegment (kategorisk prædiktor) - værdier holdes ASCII: */
      /* PLS'ens dummy-kodning afhænger af den alfabetiske rækkefølge */
      /* af niveaunavnene, så oversatte værdier ville forskyde tal-   */
      /* resultaterne i forhold til den engelske kilde                */
      u = rand('uniform');
      HVIS u < 0.34 SÅ Segment = 'Retail';
      ELLERS HVIS u < 0.70 SÅ Segment = 'Affluent';
      ELLERS Segment = 'Business';

      /* uobserverede makro-/adfærdsdrivere (kollineære) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 kollineære månedlige kreditoplysningsprædiktorer b1-b12 */
      TABEL b{12} b1-b12;
      GØR j = 1 TIL 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      SLUT;

      /* makro-kovariater, også bundet til driverne */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* tre korrelerede kreditrisiko-respons-variable */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      HVIS DefaultProp < 0 SÅ DefaultProp = 0;

      /* udelad ~25% som valideringssegment */
      HVIS rand('uniform') < 0.25 SÅ Role = 'TEST';
      ELLERS Role = 'TRAIN';

      UDDATA;
   SLUT;
   FJERN u stress tenure j;
KØR;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.36 seconds
  cpu   0.36 seconds


## Trin 2 — Tilpas multi-respons PLS-modellen med krydsvalidering

Selve kaldet demonstrerer de vigtigste `PROC HPPLS`-sætninger og de tilvalg, en risikomodellør faktisk ville bruge:

- **`MODEL`** angiver alle tre respons-variable til venstre og hele det kollineære prædiktorsæt til højre; **`/ solution`** udskriver de endelige prædiktive koefficienter på den oprindelige skala.
- **`CLASS Segment`** bringer porteføljesegmentet ind som en kategorisk prædiktor (den skal stå før `MODEL`).
- **`ID CustomerID`** bringer kunde-nøglen med til det scorede output.
- **`CVTEST(stat=press ...)`** kører van der Voet-randomiseringstesten for at vælge antallet af faktorer objektivt frem for ved øjemål; `seed=` gør det reproducerbart.
- **`PARTITION rolevar=Role(...)`** tilpasser og scorer på træningsrækkerne og udelader `TEST`-segmentet, så krydsvaliderings-PRESS måles uden for stikprøven.
- **`VARSS`** og **`DETAILS`** rapporterer, hvor meget X- og Y-variation hver efterfølgende faktor forklarer.
- **`OUTPUT`** skriver forudsagte værdier, residualer, X-scorer og PRESS for de tilpassede (trænings-)rækker til et scoret datasæt, nøglet på `CustomerID`.

In [2]:
PROCEDURE hppls data=credit nfac=8
           cvtest(stat=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   KLASSE Segment;
   id CustomerID;
   MODEL Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' test='TEST');
   UDDATA out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
KØR;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segment: 3 levels: Affluent Business Retail

Response Variable(s): Utilization DTIChange DefaultProp
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 RatePctChg InquiryCount Segment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003      1.4607 75.6872
    5      2.0832 91.9835      0.9925 76.6797
    6      1.3462 93.3297      0.8646 77.5443
    7      1.0034 94.3331      0.2783 77.8227
    8      0.6437 94.9768      0.1488 77.9714

Variation Accounted for by Variab


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Trin 3 — Opsummer den forudsagte risikoprofil

Med modellen tilpasset profilerer vi de forudsagte udfald på tværs af porteføljen. `PROC MEANS` rapporterer centraltendens og spredning for hver forudsagt respons, så risikoteamet kan sandhedsprøve skalaen (f.eks. forudsagt udnyttelsesgrad centreret i midten af 40'erne, misligholdelses-proxy nær den antagne grundrate på 8 %).

In [3]:
PROCEDURE GENNEMSNIT data=scored mean std MIN MAX maxdec=3;
   VARIABEL Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   MÆRKAT Pred_Utilization="Forudsagt udnyttelsesgrad (%)"
          Pred_DTIChange="Forudsagt ændring i gældskvote"
          Pred_DefaultProp="Forudsagt misligholdelsessandsynlighed";
KØR;

                                                  The MEANS Procedure

 Variable          Label                                            Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Forudsagt udnyttelsesgrad (%)                  47.405      10.899      29.217      82.594
 PRED_DTICHANGE    Forudsagt ændring i gældskvote                  0.649       2.503      -3.921       9.192
 PRED_DEFAULTPROP  Forudsagt misligholdelsessandsynlighed          0.090       0.049       0.008       0.235
 -----------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Trin 4 — Undersøg enkelte scorede kunder

Til sidst lister vi et par kunder fra det tilpassede **trænings**-segment med deres oprindelige `Role`-flag, forudsagt udnyttelsesgrad og residual. (De udeladte `TEST`-rækker er bevidst ikke scoret, så vi filtrerer til `Role='TRAIN'` for at vise fuldt udfyldte rækker.) Dette er den type række-niveau-output, en analytiker ville vedhæfte en modelovervågningsrapport eller sende videre til en grænsesætningsmotor.

In [4]:
PROCEDURE UDSKRIV data=scored(obs=8) MÆRKAT;
   HVOR Role = 'TRAIN';
   VARIABEL CustomerID Role Pred_Utilization Resid_Utilization;
   MÆRKAT CustomerID="Kunde-id" Role="Rolle"
          Pred_Utilization="Forudsagt udnyttelsesgrad (%)"
          Resid_Utilization="Residual, udnyttelsesgrad";
KØR;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Fortolkning af resultaterne

Tabellen **Percent Variation Accounted for** viser, at den første faktor alene optager cirka tre fjerdedele af prædiktorvariationen (74,07 %, den dominerende kollineære "stress"-retning), mens anden og tredje faktor er dem, der forklarer størstedelen af *respons*-variationen (37,94 % og 10,46 %, mod kun 25,83 % fra den første) — kendetegnet for, at PLS omorienterer komponenter mod forudsigelse frem for ren X-varians. Ved otte faktorer stabiliserer R²-værdierne pr. respons sig på 0,81 (Utilization), 0,78 (DTIChange) og 0,74 (DefaultProp), hvilket bekræfter, at de tre kreditrisiko-udfald indfanges godt af en lavdimensionel latent struktur.

Testen **van der Voet PRESS-krydsvalidering** er den afgørende faktor her: PRESS på det udeladte segment falder markant gennem de første fire faktorer (8816 → 4772 → 3318 → 3244) og flader derefter ud og stiger en smule igen, så testen vælger **fire faktorer** som den mest sparsommelige model. At udtrække flere ville jage støj i det brede kreditoplysningsblok og forringe ydeevnen uden for stikprøven — netop den overtilpasning, en kreditrisikomodel skal undgå før idriftsættelse.

`SOLUTION`-koefficienterne giver banken et fortolkeligt, regulariseret lineært scorekort på den oprindelige variabelskala, hvor `RatePctChg` (≈0,80, 0,88, 0,63 på tværs af de tre udfald) og `InquiryCount` (≈0,47, 0,36, 0,58) fremstår som de stærkeste enkeltdrivere. I praksis ville en modellør nu genberegne med `nfac=4`, overvåge residualerne i datasættet `scored` og føre faktorscorerne eller koefficienterne videre til en produktions-risikobeslutningspipeline.